# histogram_sigma_classic_control

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import lifetime_average, return_curve, subsample
from analysis.stats import best_by_mean, mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each environment's stored runs per SIGMA_RATIO, if stored.
ENVIRONMENTS = ["cartpole", "mountaincar"]
SIGMA_PATH = "AGENT_HYPERS.SIGMA_RATIO"

results = {}
for environment in ENVIRONMENTS:
    name = f"dqn_histogram_{environment}"
    for sigma in EXPERIMENT.component(name).sweep[SIGMA_PATH]:
        runs = load_results(EXPERIMENT, name, where={SIGMA_PATH: sigma})
        if runs is not None:
            results[environment, sigma] = runs

In [ ]:
# Compute each run's lifetime score, the best SIGMA_RATIO, and every learning curve.
scores = {}
curves = {}
for (environment, sigma), runs in results.items():
    returns = return_curve(runs.reward, runs.done)
    scores.setdefault(environment, {})[sigma] = lifetime_average(returns)
    timesteps, values = subsample(returns)
    curves[environment, sigma] = (timesteps, mean_ci(values))
sensitivity = {
    environment: {sigma: mean_ci(runs) for sigma, runs in by_sigma.items()}
    for environment, by_sigma in scores.items()
}
best_sigma = {
    environment: best_by_mean(by_sigma) for environment, by_sigma in scores.items()
}
for environment, sigma in best_sigma.items():
    print(f"[{environment}] best SIGMA_RATIO: {sigma}")

In [ ]:
# Style for the SIGMA_RATIO sensitivity and learning-curve plots.
STYLE = {
    "figure.figsize": (9, 6),
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
COLOR = "tab:blue"
BAND_ALPHA = 0.2
TITLES = {"cartpole": "Cartpole", "mountaincar": "MountainCar"}
YLIMS = {"cartpole": (0, 500), "mountaincar": (-1000, 0)}
PLOT_TITLE = "HL-Gauss DQN on {title} ({seeds} seeds, 95% CI)"
YLABEL_KW = {"rotation": 0, "ha": "right", "va": "center", "labelpad": 12}
SENSITIVITY_LABEL = "HL-Gauss DQN (best σ/bin={sigma})"
SENSITIVITY_XLABEL = "SIGMA_RATIO (σ / bin width)"
SENSITIVITY_YLABEL = "Weighted\nlifetime\nreturn"
SENSITIVITY_LEGEND = {"loc": "best"}
CURVE_LABEL = "σ/bin = {sigma}"
CURVE_XLABEL = "Timestep"
CURVE_YLABEL = "Return"
CURVE_LEGEND = {"loc": "best"}


def sigma_colors(sigmas):
    """One viridis colour per SIGMA_RATIO, in order, so the sweep reads as a gradient."""
    return dict(zip(sigmas, plt.cm.viridis(np.linspace(0, 0.9, len(sigmas))), strict=True))


def seeds(environment):
    return len(next(iter(scores[environment].values())))

In [ ]:
# Plot each environment's mean lifetime return against SIGMA_RATIO.
figures = {}
for environment in ENVIRONMENTS:
    if environment not in sensitivity:
        continue
    sigmas = sorted(sensitivity[environment])
    mean, low, high = np.array([sensitivity[environment][s] for s in sigmas]).T
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        ax.fill_between(sigmas, low, high, color=COLOR, alpha=BAND_ALPHA)
        label = SENSITIVITY_LABEL.format(sigma=best_sigma[environment])
        ax.plot(sigmas, mean, color=COLOR, marker="o", label=label)
        ax.set_xticks(sigmas)
        title = PLOT_TITLE.format(title=TITLES[environment], seeds=seeds(environment))
        ax.set(title=title, xlabel=SENSITIVITY_XLABEL)
        ax.set_ylabel(SENSITIVITY_YLABEL, **YLABEL_KW)
        ax.legend(**SENSITIVITY_LEGEND)
        fig.tight_layout()
    figures[f"{environment}_sigma_sensitivity"] = fig
plt.show()

In [ ]:
# Plot each environment's learning curves, one per SIGMA_RATIO.
for environment in ENVIRONMENTS:
    sigmas = sorted(sigma for env, sigma in curves if env == environment)
    if not sigmas:
        continue
    colors = sigma_colors(sigmas)
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for sigma in sigmas:
            timesteps, (mean, low, high) = curves[environment, sigma]
            ax.fill_between(timesteps, low, high, color=colors[sigma], alpha=BAND_ALPHA)
            ax.plot(timesteps, mean, color=colors[sigma], label=CURVE_LABEL.format(sigma=sigma))
        title = PLOT_TITLE.format(title=TITLES[environment], seeds=seeds(environment))
        ax.set(title=title, xlabel=CURVE_XLABEL, ylim=YLIMS[environment])
        ax.set_ylabel(CURVE_YLABEL, **YLABEL_KW)
        ax.set_xticks([0, timesteps[-1]])  # timestep axis: show only 0 and the max
        ax.legend(**CURVE_LEGEND)
        fig.tight_layout()
    figures[f"{environment}_learning_curves"] = fig
plt.show()

In [ ]:
# Save every plotted figure as a PDF and a PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(PLOTS_DIR / f"{name}.png", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")